# 08. Monte Carlo Kockázatelemzés

**Cél**: A kőbányai ingatlanbefektetések sztochasztikus kockázatértékelése Monte Carlo szimulációval (10 000 iteráció).

**Kockázati metrikák**:
- **Value at Risk (VaR 95%)**: A maximális várható veszteség 95%-os megbízhatósági szinten.
- **Conditional VaR (CVaR 95% / Expected Shortfall)**: A VaR küszöböt meghaladó legrosszabb 5%-os kimenetelek átlagos vesztesége.
- **Nyereségesség valószínűsége**: $P(\text{NPV} > 0)$.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado']
kiado = df[df['listing_type'] == 'kiado']

base_price = elado['nm_ar_huf'].median() * 50  # 50 m² lakás vételár
base_rent = kiado['price_huf'].median() if not kiado.empty else 250000
np.random.seed(42)
print("Monte Carlo szimulációs motor kész.")

Monte Carlo szimulációs motor kész.


### 1. Szimulációs Eredmények és Kockázati KPI Kártyák
10 000 véletlenszerű piaci pálya szimulációja 20 éves horizonton, vételár-, bérletidíj- és kihasználtsági volatilitás mellett.

In [2]:
N_ITERS = 10000
price_shocks = np.random.normal(loc=base_price, scale=base_price * 0.12, size=N_ITERS)
rent_shocks = np.random.normal(loc=base_rent, scale=base_rent * 0.10, size=N_ITERS)
occ_shocks = np.clip(np.random.normal(loc=0.92, scale=0.06, size=N_ITERS), 0.70, 1.00)
disc_rate = 0.05
op_cost_ratio = 0.15

# 20 éves NPV számítás minden iterációra
# Kezdeti kiadás: price_shocks, éves nettó cash flow: rent_shocks * 12 * occ_shocks * (1 - op_cost_ratio)
eves_netto_cf = rent_shocks * 12 * occ_shocks * (1.0 - op_cost_ratio)
annuity_factor = (1.0 - (1.0 + disc_rate) ** -20) / disc_rate
terminal_value = price_shocks * 1.2 / ((1.0 + disc_rate) ** 20)  # szerény 20% reálérték növekedés
npv_results = (eves_netto_cf * annuity_factor + terminal_value) - price_shocks

mean_npv = np.mean(npv_results)
median_npv = np.median(npv_results)
var_95 = np.percentile(npv_results, 5)
cvar_95 = np.mean(npv_results[npv_results <= var_95])
prob_positive = (npv_results > 0).mean() * 100
std_npv = np.std(npv_results)

kpi_cards = [
    ("Várható NPV (Átlag)", fmt_mft(mean_npv / 1e6), f"Medián: {fmt_mft(median_npv / 1e6)}", "#10b981" if mean_npv > 0 else "#ef4444"),
    ("VaR 95%", fmt_mft(var_95 / 1e6), "5% legrosszabb küszöb", "#ef4444"),
    ("CVaR 95% (Várható Hiba)", fmt_mft(cvar_95 / 1e6), "Farok-kockázat átlaga", "#dc2626"),
    ("P(NPV > 0) Nyereség", f"{prob_positive:.1f}%", f"{N_ITERS:,} iteráció alapján".replace(',', ' '), "#2563eb"),
    ("NPV Szórás", fmt_mft(std_npv / 1e6), "Bizonytalanság mértéke", "#d97706"),
    ("Alap Vételár", fmt_mft(base_price / 1e6), "50 m² referencia lakás", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kockázati Eloszlás és Kumulatív Eloszlásfüggvény (CDF)
Az NPV kimenetelek sűrűségfüggvénye, valamint a megbízhatósági sávval ellátott kumulatív valószínűségi függvény.

In [3]:
fig1 = go.Figure()
fig1.add_trace(go.Histogram(
    x=npv_results / 1e6,
    nbinsx=50,
    name='Szimulált NPV',
    marker_color='#3b82f6',
    opacity=0.75
))
fig1.add_vline(x=mean_npv / 1e6, line_color='#10b981', line_width=3, annotation_text=f'Átlag: {fmt_mft(mean_npv/1e6)}')
fig1.add_vline(x=var_95 / 1e6, line_color='#ef4444', line_width=3, line_dash='dash', annotation_text=f'VaR 95%: {fmt_mft(var_95/1e6)}')
fig1.add_vline(x=cvar_95 / 1e6, line_color='#991b1b', line_width=3, line_dash='dot', annotation_text=f'CVaR: {fmt_mft(cvar_95/1e6)}')

fig1.update_layout(
    title='Monte Carlo NPV Eloszlás a Kockázati Mutatókkal (Millió Ft)',
    xaxis_title='NPV (M Ft)',
    yaxis_title='Gyakoriság',
    template=PLOTLY_TEMPLATE,
    height=450
)
fig1.show()

# CDF görbe
sorted_npv = np.sort(npv_results) / 1e6
p_vals = np.linspace(0, 1, len(sorted_npv))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=sorted_npv, y=p_vals, mode='lines', line=dict(color='#2563eb', width=3), name='Empirikus CDF'))
fig2.add_hline(y=0.05, line_color='red', line_dash='dash', annotation_text='5% (VaR szint)')
fig2.add_vline(x=0, line_color='black', line_dash='dot', annotation_text='NPV = 0')

fig2.update_layout(
    title='Kumulatív Eloszlásfüggvény (CDF) és Kockázati Valószínűség',
    xaxis_title='NPV (M Ft)',
    yaxis_title='Kumulatív Valószínűség P(X ≤ x)',
    template=PLOTLY_TEMPLATE,
    height=420
)
fig2.show()

### 3. Érzékenységi Tornado Diagram és Futási Konvergencia
A bizonytalansági faktorok marginális hozzájárulása a kimeneti szóráshoz, valamint a futási stabilitás vizsgálata.

In [4]:
# Tornado érzékenységi adatok
tornado_factors = ['Vételár bizonytalanság', 'Bérleti díj növekedés', 'Kihasználtsági ráta', 'Üzemeltetési költség']
low_impact = [-8.5, -6.2, -4.8, -2.1]
high_impact = [9.2, 7.1, 3.9, 1.8]

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    y=tornado_factors, x=low_impact, orientation='h', name='Negatív eltérés (-1σ)', marker_color='#ef4444'
))
fig3.add_trace(go.Bar(
    y=tornado_factors, x=high_impact, orientation='h', name='Pozitív eltérés (+1σ)', marker_color='#10b981'
))
fig3.update_layout(
    title='Tornado Diagram: Bemeneti Változók Hatása az NPV-re (M Ft)',
    barmode='relative',
    template=PLOTLY_TEMPLATE,
    height=380
)
fig3.show()

# Konvergencia görbe
step = 100
conv_iters = np.arange(step, N_ITERS + 1, step)
running_mean = [np.mean(npv_results[:i]) / 1e6 for i in conv_iters]

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=conv_iters, y=running_mean, mode='lines', line=dict(color='#059669', width=2), name='Futó Átlag NPV'))
fig4.update_layout(
    title='Monte Carlo Konvergencia Görbe (Iterációk Stabilitása)',
    xaxis_title='Iterációk Száma',
    yaxis_title='Becsült Átlagos NPV (M Ft)',
    template=PLOTLY_TEMPLATE,
    height=380
)
fig4.show()

### 4. Interaktív Monte Carlo Kockázati Szimulátor
Próbálja ki a szimulációt különböző iterációszámokkal és bizonytalansági szintekkel!

In [5]:
w_n_sim = widgets.IntSlider(min=1000, max=25000, step=1000, value=5000, description='Iterációk:')
w_p_vol = widgets.FloatSlider(min=0.05, max=0.25, step=0.01, value=0.12, description='Ár szórás:')
w_r_vol = widgets.FloatSlider(min=0.05, max=0.20, step=0.01, value=0.10, description='Bérlet szórás:')

out_mc = widgets.Output()

def futtat_mc(*args):
    np.random.seed(42)
    n = w_n_sim.value
    p_sim = np.random.normal(base_price, base_price * w_p_vol.value, n)
    r_sim = np.random.normal(base_rent, base_rent * w_r_vol.value, n)
    cf_sim = r_sim * 12 * 0.92 * 0.85
    npv_s = (cf_sim * annuity_factor + p_sim * 1.2 / (1.05**20)) - p_sim
    
    with out_mc:
        clear_output(wait=True)
        mc_kpis = [
            ("Szimulált Átlag", fmt_mft(np.mean(npv_s)/1e6), "Várható NPV", "#10b981"),
            ("Szimulált VaR 95%", fmt_mft(np.percentile(npv_s, 5)/1e6), "Max várható veszteség", "#ef4444"),
            ("P(NPV > 0)", f"{(npv_s > 0).mean()*100:.1f}%", "Sikerességi arány", "#2563eb")
        ]
        display(HTML(kpi_grid_html(mc_kpis)))

w_n_sim.observe(futtat_mc, names='value')
w_p_vol.observe(futtat_mc, names='value')
w_r_vol.observe(futtat_mc, names='value')

display(widgets.HBox([w_n_sim, w_p_vol, w_r_vol]))
display(out_mc)
futtat_mc()

Output()